In [ ]:
# Open from GeoTest. Close other training jobs for meaningful timings.
# 8 trials by default: 2 modalities x 4 worker counts, each trained for 10 epochs.
# Each trial compares evaluation batches using the SAME final model on validation.
# No test-set use, checkpoint saving, or modifications to scientific configurations.
from pathlib import Path
from datetime import datetime
import csv, json, os, random, subprocess, sys, uuid

CONFIGS = {
    'S1': 'configs/croma/croma_s1_fcn_all_hand_labels.yaml',
    'S1S2': 'configs/croma/croma_s1s2_fcn_all_hand_labels.yaml',
}
WORKERS = [0, 2, 4, 8]
EVAL_BATCH_SIZES = [1, 4, 8]
TRAIN_BATCH_SIZE = 16
TILE_BATCH_SIZE = 16  # Fixed to isolate full-image batch size and workers.
EVAL_REPEATS = 2
VALIDATE_EVERY = 10  # For projected runtime ranking only.
assert Path('run.py').is_file(), 'Use GeoTest as the notebook working directory.'
assert WORKERS and all(type(w) is int and w >= 0 for w in WORKERS)
assert EVAL_BATCH_SIZES and all(type(b) is int and b > 0 for b in EVAL_BATCH_SIZES)
assert min(TRAIN_BATCH_SIZE,TILE_BATCH_SIZE,EVAL_REPEATS,VALIDATE_EVERY)>0
BENCH_DIR = Path('runs/benchmarks/croma_throughput') / (datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
BENCH_DIR.mkdir(parents=True,exist_ok=False)
print('Results:',BENCH_DIR)
print(len(CONFIGS)*len(WORKERS),'training trials, 10 epochs each')


In [ ]:
# Generate a guarded worker script in the results folder for Windows multiprocessing.
WORKER_SOURCE = 'import json, random, statistics, sys, time, traceback\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom utils import load_config, build_preprocessing, build_loaders, build_loss, build_optimizer, build_scheduler, write_json\nfrom src.models.factory import build_model\nfrom training.trainer import train_one_epoch\nfrom src.evaluation.evaluator import evaluate\n\ndef main():\n    job=json.loads(Path(sys.argv[1]).read_text())\n    out=Path(job[\'result\'])\n    result=dict(job=job, status=\'running\', training=[], evaluation=[])\n    write_json(out,result)\n    try:\n        cfg=load_config(job[\'yaml\'])\n        cfg[\'dataset\'].pop(\'test_csv\',None)\n        cfg[\'dataset\'].pop(\'bolivia_csv\',None)\n        cfg[\'training\'].update(epochs=10,batch_size=job[\'train_batch\'],num_workers=job[\'workers\'])\n        cfg[\'evaluation\'].update(batch_size=1,tile_batch_size=job[\'tile_batch\'])\n        result[\'config\']=cfg\n        seed=cfg[\'experiment\'].get(\'seed\',234)\n        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        def sync():\n            if device.type==\'cuda\': torch.cuda.synchronize()\n        result[\'device\']=str(device)\n        result[\'gpu\']=torch.cuda.get_device_name() if device.type==\'cuda\' else None\n        result[\'torch_version\']=torch.__version__\n        start=time.perf_counter()\n        normalizer,_=build_preprocessing(cfg)\n        loaders=build_loaders(cfg,normalizer)\n        model=build_model(cfg).to(device)\n        criterion=build_loss(cfg[\'training\'][\'loss\'],device)\n        optimizer=build_optimizer(cfg[\'training\'][\'optimizer\'],model)\n        scheduler=build_scheduler(cfg[\'training\'].get(\'scheduler\'),optimizer,len(loaders[\'train\']))\n        sync(); result[\'setup_seconds\']=time.perf_counter()-start\n        for epoch in range(1,11):\n            sync(); start=time.perf_counter()\n            loss=train_one_epoch(model,loaders[\'train\'],optimizer,criterion,device,scheduler=scheduler,input_keys=cfg[\'dataset\'][\'modalities\'])\n            sync(); elapsed=time.perf_counter()-start\n            result[\'training\'].append(dict(epoch=epoch,seconds=elapsed,loss=loss))\n            write_json(out,result)\n            print(f"{job[\'modality\']} workers={job[\'workers\']} epoch={epoch}/10: {elapsed:.2f}s",flush=True)\n        result[\'train_median_seconds\']=statistics.median(r[\'seconds\'] for r in result[\'training\'][1:])\n        batches=list(job[\'eval_batches\'])\n        random.Random(seed+job[\'workers\']).shuffle(batches)\n        reference=None\n        for batch in batches:\n            row=dict(batch_size=batch,status=\'running\',seconds=[])\n            result[\'evaluation\'].append(row)\n            try:\n                cfg[\'evaluation\'][\'batch_size\']=batch\n                loader=build_loaders(cfg,normalizer)[\'val\']\n                options=cfg[\'evaluation\']\n                if device.type==\'cuda\': torch.cuda.reset_peak_memory_stats()\n                for repeat in range(job[\'repeats\']):\n                    sync(); start=time.perf_counter()\n                    metrics=evaluate(model,loader,device,num_classes=cfg[\'head\'][\'num_classes\'],criterion=criterion,\n                        tile_size=options.get(\'tile_size\'),stride=options.get(\'stride\'),tile_batch_size=job[\'tile_batch\'],\n                        input_keys=cfg[\'dataset\'][\'modalities\'],return_per_sample=options.get(\'return_per_sample\',True),\n                        sample_id_key=options.get(\'sample_id_key\',\'chip_id\'),class_names=options.get(\'class_names\'),\n                        aw_pw_fw=options.get(\'aw_pw_fw\',False))\n                    sync(); elapsed=time.perf_counter()-start\n                    row[\'seconds\'].append(elapsed)\n                    row[\'metrics\']={k:v for k,v in metrics.items() if k!=\'per_sample\'}\n                    if reference is None: reference=metrics[\'miou\']\n                    row[\'miou_difference_from_first\']=metrics[\'miou\']-reference\n                    print(f"  evaluation batch={batch} repeat={repeat+1}: {elapsed:.2f}s",flush=True)\n                    write_json(out,result)\n                row[\'median_seconds\']=statistics.median(row[\'seconds\'])\n                row[\'peak_gpu_GiB\']=torch.cuda.max_memory_allocated()/2**30 if device.type==\'cuda\' else None\n                row[\'status\']=\'complete\'\n            except torch.cuda.OutOfMemoryError:\n                row[\'status\']=\'out_of_memory\'\n                row[\'error\']=traceback.format_exc()\n                torch.cuda.empty_cache()\n            write_json(out,result)\n        result[\'status\']=\'complete\'\n    except Exception:\n        result[\'status\']=\'failed\'\n        result[\'error\']=traceback.format_exc()\n        print(result[\'error\'],flush=True)\n    finally: write_json(out,result)\n\nif __name__==\'__main__\': main()\n'
WORKER_SCRIPT = BENCH_DIR / "benchmark_worker.py"
WORKER_SCRIPT.write_text(WORKER_SOURCE,encoding="utf-8")


In [ ]:
# Sequential trials prevent GPU contention. Logs stream here and are saved.
# Re-running this cell skips completed trials; incomplete trials restart.
jobs=[(name,path,w) for name,path in CONFIGS.items() for w in WORKERS]
random.Random(234).shuffle(jobs)
env=os.environ.copy()
env['PYTHONPATH']=str(Path.cwd())+os.pathsep+env.get('PYTHONPATH','')
for name,path,w in jobs:
    result=BENCH_DIR/f'{name}_workers{w}.json'
    if result.exists() and json.loads(result.read_text())['status']=='complete':
        print('Already completed:',result.name)
        continue
    job=dict(modality=name,yaml=path,workers=w,train_batch=TRAIN_BATCH_SIZE,
             tile_batch=TILE_BATCH_SIZE,eval_batches=EVAL_BATCH_SIZES,repeats=EVAL_REPEATS,result=str(result))
    job_path=BENCH_DIR/f'{name}_workers{w}_job.json'
    job_path.write_text(json.dumps(job,indent=2),encoding='utf-8')
    with (BENCH_DIR/f'{name}_workers{w}.log').open('w',encoding='utf-8') as log:
        process=subprocess.Popen([sys.executable,'-u',str(WORKER_SCRIPT),str(job_path)],
            cwd=Path.cwd(),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,encoding='utf-8',errors='replace')
        try:
            for line in process.stdout:
                log.write(line); log.flush(); print(line,end='')
            code=process.wait()
            if code: print('Process failed; inspect log. Exit code:',code)
        except KeyboardInterrupt:
            if os.name=='nt':
                subprocess.run(['taskkill','/PID',str(process.pid),'/T','/F'],capture_output=True)
            else: process.terminate()
            process.wait()
            raise


In [ ]:
# Median training time excludes epoch 1; loader startup remains included.
# Cache warming and worker-dependent random crops limit exact comparisons.
# Rankings exclude setup, checkpoint writes and test evaluation.
rows=[]
failures=[]
for name in CONFIGS:
    for w in WORKERS:
        path=BENCH_DIR/f'{name}_workers{w}.json'
        if not path.exists(): continue
        trial=json.loads(path.read_text())
        if trial['status']!='complete': failures.append(dict(modality=name,workers=w,status=trial['status']))
        for evaluation in trial['evaluation']:
            if evaluation['status']!='complete':
                failures.append(dict(modality=name,workers=w,batch=evaluation['batch_size'],status=evaluation['status']))
                continue
            train=trial['train_median_seconds']
            val=evaluation['median_seconds']
            rows.append(dict(modality=name,num_workers=w,eval_batch_size=evaluation['batch_size'],
                train_epoch_seconds=train,validation_seconds=val,
                projected_seconds_per_epoch=train+val/VALIDATE_EVERY,
                peak_eval_gpu_GiB=evaluation['peak_gpu_GiB'],miou_difference=evaluation['miou_difference_from_first']))
rows.sort(key=lambda r:(r['modality'],r['projected_seconds_per_epoch']))
if rows:
    with (BENCH_DIR/'ranking.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
best={name:next((r for r in rows if r['modality']==name),None) for name in CONFIGS}
(BENCH_DIR/'summary.json').write_text(json.dumps(dict(best_measured=best,rankings=rows,failures=failures,
    validate_every_projection=VALIDATE_EVERY),indent=2),encoding='utf-8')
for name,recommendation in best.items(): print(name,recommendation)
print('Failures:',failures)
print('Results:',BENCH_DIR)


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(len(CONFIGS),2,figsize=(12,4*len(CONFIGS)),squeeze=False)
for i,name in enumerate(CONFIGS):
    selected=[r for r in rows if r['modality']==name]
    times={r['num_workers']:r['train_epoch_seconds'] for r in selected}
    workers=sorted(times)
    axes[i,0].plot(workers,[times[w] for w in workers],marker='o')
    axes[i,0].set(title=name+' training',xlabel='Workers',ylabel='Seconds / epoch')
    for batch in EVAL_BATCH_SIZES:
        values=sorted([r for r in selected if r['eval_batch_size']==batch],key=lambda r:r['num_workers'])
        axes[i,1].plot([r['num_workers'] for r in values],[r['validation_seconds'] for r in values],marker='o',label=f'Batch {batch}')
    axes[i,1].set(title=name+' evaluation',xlabel='Workers',ylabel='Seconds / validation')
    if selected: axes[i,1].legend()
fig.tight_layout()
fig.savefig(BENCH_DIR/'timing_comparison.png',dpi=150)
plt.show()
